In [8]:
# ------------------------------------------------------------
# calibrate_in_notebook.py
# ------------------------------------------------------------
# Compute optimal logistic scale (s) for each season's Colley ratings.
# Designed for Jupyter Notebook use — no argparse or CLI required.
#
# Example usage (at bottom of file):
#
# df, s_mean = calibrate_by_season_notebook(
#     teams_path="NCAA_2025_Teams.txt",
#     games_glob="NCAA_*_Games.txt",
#     avg_spl_path="avg_spl_2025.json",
#     segmentWeighting=(0.5, 1, 2),
#     useWeighting=True
# )
#
# ------------------------------------------------------------

import glob
import json
import math
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Utility functions ----------
def load_teams(teams_path: str) -> pd.DataFrame:
    """Parse teams file like '   1, Abilene_Chr' -> DataFrame (0-based ids)."""
    with open(teams_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.read().splitlines()

    rows = []
    for ln in lines:
        m = re.match(r"^\s*(\d+)\s*,\s*(.+?)\s*$", ln.strip())
        if m:
            one_based = int(m.group(1))
            rows.append((one_based - 1, m.group(2)))

    if not rows:
        raise ValueError("Could not parse teams file — expected '1, TeamName' style lines")

    df = pd.DataFrame(rows, columns=["team_id", "team_name"])
    return df.sort_values("team_id", ignore_index=True)


def read_games(path: str) -> pd.DataFrame:
    """Games file: 8 columns, no header."""
    df = pd.read_csv(path, header=None)
    if df.shape[1] < 8:
        raise ValueError(f"{path}: expected 8 columns, got {df.shape[1]}")
    return df


def year_from_games(df: pd.DataFrame) -> int:
    """Infer season year from column 1 (YYYYMMDD)."""
    return int(df.iloc[0, 1]) // 10000


def logistic(x: np.ndarray) -> np.ndarray:
    """Logistic function with clipping for numerical safety."""
    x = np.clip(x, -60, 60)
    return 1.0 / (1.0 + np.exp(-x))


# ---------- Colley method (exact version) ----------
def compute_colley_exact(
    games: pd.DataFrame,
    num_teams: int,
    avg_spl: dict,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
) -> tuple[np.ndarray, float]:
    """Compute Colley ratings exactly as in Austin’s original code."""
    colleyMatrix = 2 * np.diag(np.ones(num_teams))
    b = np.ones(num_teams)

    def weight_avg_spl(x: int) -> float:
        return 3 * math.log10(1 / (avg_spl[x] - 2)) + 1.75

    numGames = len(games)
    dayBeforeSeason = games.iloc[0, 0] - 1
    lastDayOfSeason = games.iloc[numGames - 1, 0]

    for i in range(numGames):
        team1ID = int(games.iloc[i, 2]) - 1
        team1Score = float(games.iloc[i, 4])
        team2ID = int(games.iloc[i, 5]) - 1
        team2Score = float(games.iloc[i, 7])
        currentDay = float(games.iloc[i, 0])

        if useWeighting:
            numberSegments = len(segmentWeighting)
            weightIndex = math.ceil(
                numberSegments * ((currentDay - dayBeforeSeason) / (lastDayOfSeason - dayBeforeSeason))
            ) - 1
            timeWeight = float(segmentWeighting[weightIndex])
        else:
            timeWeight = 1.0

        used = False
        if team1Score > team2Score and team2ID+1 in avg_spl:
            gameWeight = weight_avg_spl(team2ID+1) * timeWeight
            used = True
        elif team2Score > team1Score and team1ID+1 in avg_spl:
            gameWeight = weight_avg_spl(team1ID+1) * timeWeight
            used = True

        if not used:
            continue

        # Update matrix
        colleyMatrix[team1ID, team2ID] -= gameWeight
        colleyMatrix[team2ID, team1ID] -= gameWeight
        colleyMatrix[team1ID, team1ID] += gameWeight
        colleyMatrix[team2ID, team2ID] += gameWeight

        # RHS updates
        if team1Score > team2Score:
            b[team1ID] += 0.5 * gameWeight
            b[team2ID] -= 0.5 * gameWeight
        elif team2Score > team1Score:
            b[team1ID] -= 0.5 * gameWeight
            b[team2ID] += 0.5 * gameWeight

    r = np.linalg.solve(colleyMatrix, b)

    # Predictability
    correct = 0
    for i in range(numGames):
        t1 = int(games.iloc[i, 2]) - 1
        s1 = float(games.iloc[i, 4])
        t2 = int(games.iloc[i, 5]) - 1
        s2 = float(games.iloc[i, 7])
        if (s1 > s2 and r[t1] > r[t2]) or (s2 > s1 and r[t2] > r[t1]) or (s1 == s2 and r[t1] == r[t2]):
            correct += 1
    predictability = 100.0 * correct / numGames if numGames else 0.0
    return r, predictability


# ---------- Fit logistic scale s ----------
def fit_scale_for_games(
    r: np.ndarray,
    games: pd.DataFrame,
    s_min=1.0,
    s_max=16.0,
    steps=301,
):
    """Find best logistic scale s minimizing log-loss."""
    t1 = games.iloc[:, 2].to_numpy(dtype=int) - 1
    s1 = games.iloc[:, 4].to_numpy(dtype=float)
    t2 = games.iloc[:, 5].to_numpy(dtype=int) - 1
    s2 = games.iloc[:, 7].to_numpy(dtype=float)
    y = (s1 > s2).astype(float) + 0.5 * (s1 == s2)

    dR = r[t1] - r[t2]
    s_values = np.linspace(s_min, s_max, steps)
    losses = []

    for s in s_values:
        p = logistic(dR * s)
        p = np.clip(p, 1e-9, 1 - 1e-9)
        loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        losses.append(loss)

    best_idx = int(np.argmin(losses))
    return float(s_values[best_idx]), float(losses[best_idx]), s_values, np.array(losses)


# ---------- Main calibration workflow ----------
def calibrate_by_season_notebook(
    teams_path: str,
    games_glob: str,
    avg_spl: dict | str,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
    s_min=1.0,
    s_max=16.0,
    steps=301,
    plot_dir=None,
):
    """Loop over all seasons, compute Colley + fit s, return DataFrame and weighted mean."""
    teams_df = load_teams(teams_path)
    num_teams = len(teams_df)

    # Accept dict OR JSON path
    if isinstance(avg_spl, str):
        with open(avg_spl, "r") as f:
            raw = json.load(f)
        avg_spl = {int(k): float(v) for k, v in raw.items()}
    elif isinstance(avg_spl, dict):
        avg_spl = {int(k): float(v) for k, v in avg_spl.items()}
    else:
        raise ValueError("avg_spl must be dict or path to JSON file")

    files = sorted(glob.glob(games_glob))
    if not files:
        raise FileNotFoundError(f"No files matched {games_glob}")

    results = []
    if plot_dir:
        os.makedirs(plot_dir, exist_ok=True)

    for gpath in files:
        games = read_games(gpath)
        yr = year_from_games(games)

        r, predict = compute_colley_exact(
            games,
            num_teams,
            avg_spl,
            segmentWeighting=segmentWeighting,
            useWeighting=useWeighting,
        )
        best_s, best_loss, s_vals, losses = fit_scale_for_games(
            r, games, s_min=s_min, s_max=s_max, steps=steps
        )

        results.append((yr, best_s, best_loss, len(games), predict))
        print(f"[{yr}] best s={best_s:.3f}, logloss={best_loss:.6f}, predict={predict:.2f}%")

        if plot_dir:
            plt.figure()
            plt.plot(s_vals, losses, "-")
            plt.axvline(best_s, color="r", linestyle="--", label=f"best s={best_s:.2f}")
            plt.xlabel("scale (s)")
            plt.ylabel("log-loss")
            plt.title(f"{yr} calibration curve")
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f"scale_{yr}.png"), dpi=150)
            plt.close()

    df = pd.DataFrame(results, columns=["year", "best_s", "logloss", "n_games", "predictability"])
    df["weighted"] = df["best_s"] * df["n_games"]
    weighted_mean = df["weighted"].sum() / df["n_games"].sum()
    print("\nWeighted mean s =", round(weighted_mean, 3))

    return df.sort_values("year"), weighted_mean

import numpy as np
import pandas as pd

def recompute_logloss(r, games_df, s):
    t1 = games_df.iloc[:, 2].to_numpy(int) - 1
    s1 = games_df.iloc[:, 4].to_numpy(float)
    t2 = games_df.iloc[:, 5].to_numpy(int) - 1
    s2 = games_df.iloc[:, 7].to_numpy(float)
    y = (s1 > s2).astype(float) + 0.5 * (s1 == s2)

    dR = r[t1] - r[t2]
    p = 1.0 / (1.0 + np.exp(-np.clip(dR * s, -60, 60)))
    p = np.clip(p, 1e-9, 1 - 1e-9)

    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    acc  = np.mean(((p >= 0.5) & (y == 1.0)) | ((p < 0.5) & (y == 0.0)) | (y == 0.5))
    return loss, acc
 
avg_spl ={
    327: 2.31714285714286,
    167: 2.34285714285714,
    170: 2.34571428571429,
    205: 2.35714285714286,
    75: 2.38285714285714,
    349: 2.40857142857143,
    235: 2.41142857142857,
    139: 2.41428571428571,
    135: 2.43142857142857,
    218: 2.43428571428571,
    344: 2.43714285714286,
    338: 2.44,
    222: 2.45142857142857,
    291: 2.45142857142857,
    128: 2.45428571428571,
    67: 2.45714285714286,
    317: 2.46285714285714,
    161: 2.47142857142857,
    266: 2.47428571428571,
    283: 2.47428571428571,
    290: 2.47714285714286,
    264: 2.48,
    326: 2.48,
    150: 2.48571428571429,
    293: 2.49142857142857,
    104: 2.50285714285714,
    97: 2.50571428571429,
    38: 2.51714285714286,
    228: 2.51714285714286,
    124: 2.52285714285714,
    57: 2.53428571428571,
    127: 2.53714285714286,
    4: 2.54285714285714,
    276: 2.54285714285714,
    54: 2.54571428571429,
    93: 2.54571428571429,
    115: 2.54857142857143,
    214: 2.55142857142857,
    12: 2.55714285714286,
    169: 2.55714285714286,
    234: 2.55714285714286,
    47: 2.56857142857143,
    159: 2.56857142857143,
    223: 2.57714285714286,
    261: 2.58,
    34: 2.58285714285714,
    299: 2.58285714285714,
    154: 2.58571428571429,
    90: 2.58857142857143,
    217: 2.58857142857143,
    324: 2.59714285714286,
    328: 2.59714285714286,
    322: 2.61142857142857,
    211: 2.62285714285714,
    240: 2.62285714285714,
    314: 2.62285714285714,
    193: 2.62571428571429,
    66: 2.62857142857143,
    136: 2.62857142857143,
    272: 2.62857142857143,
    335: 2.63714285714286,
    49: 2.64285714285714,
    102: 2.64285714285714,
    280: 2.65142857142857,
    11: 2.65428571428571,
    21: 2.66571428571429,
    172: 2.66571428571429,
    238: 2.66571428571429,
    212: 2.66857142857143,
    287: 2.67142857142857,
    306: 2.67428571428571,
    226: 2.67714285714286,
    286: 2.68,
    60: 2.68571428571429,
    101: 2.69142857142857,
    105: 2.69142857142857,
    323: 2.69714285714286,
    177: 2.70285714285714,
    341: 2.70857142857143,
    195: 2.71142857142857,
    245: 2.71142857142857,
    334: 2.71142857142857,
    165: 2.71428571428571,
    134: 2.72,
    246: 2.72571428571429,
    15: 2.73142857142857,
    219: 2.73142857142857,
    121: 2.73714285714286,
    35: 2.74,
    173: 2.74857142857143,
    3: 2.75142857142857,
    45: 2.76,
    313: 2.76857142857143,
    76: 2.77142857142857,
    111: 2.78571428571429,
    209: 2.79142857142857,
    18: 2.79428571428571,
    125: 2.79714285714286,
    301: 2.79714285714286,
    100: 2.8,
    22: 2.80285714285714,
    13: 2.81428571428571,
    285: 2.81428571428571,
    251: 2.81714285714286,
    107: 2.82,
    278: 2.82,
    340: 2.82285714285714,
    71: 2.82571428571429,
    52: 2.84857142857143,
    113: 2.85142857142857,
    184: 2.85142857142857,
    315: 2.85714285714286,
    174: 2.86285714285714,
    162: 2.87142857142857,
    138: 2.87714285714286,
    215: 2.87714285714286,
    146: 2.88,
    166: 2.88571428571429,
    171: 2.88571428571429,
    227: 2.88571428571429,
    312: 2.88571428571429,
    149: 2.88857142857143,
    198: 2.88857142857143,
    85: 2.89142857142857,
    129: 2.89428571428571,
    221: 2.89428571428571,
    345: 2.89714285714286,
    220: 2.9,
    153: 2.90285714285714,
    33: 2.90571428571429,
    122: 2.90571428571429,
    55: 2.90857142857143,
    84: 2.90857142857143,
    277: 2.91142857142857,
    342: 2.91714285714286,
    25: 2.92571428571429,
    216: 2.92857142857143,
    110: 2.93142857142857,
    168: 2.93714285714286,
    295: 2.94285714285714,
    303: 2.94857142857143,
    94: 2.95142857142857,
    95: 2.96285714285714,
    298: 2.96571428571429,
    294: 2.96857142857143,
    296: 2.96857142857143,
    83: 2.97428571428571,
    180: 2.97428571428571,
    254: 2.97428571428571,
    20: 2.99142857142857,
    185: 2.99428571428571,
    289: 2.99428571428571,
    81: 2.99714285714286,
    339: 3.00285714285714,
    207: 3.00571428571429,
    188: 3.00857142857143,
    347: 3.00857142857143,
    17: 3.01142857142857,
    176: 3.01428571428571,
    199: 3.01428571428571,
    320: 3.01714285714286,
    2: 3.02285714285714,
    140: 3.02571428571429,
    194: 3.02857142857143,
    160: 3.04,
    187: 3.04,
    267: 3.04,
    263: 3.05142857142857,
    310: 3.05142857142857,
    61: 3.05428571428571,
    237: 3.05714285714286,
    130: 3.06,
    196: 3.06285714285714,
    19: 3.07142857142857,
    72: 3.07428571428571,
    332: 3.07428571428571,
    331: 3.07714285714286,
    78: 3.08,
    89: 3.08285714285714,
    249: 3.08285714285714,
    268: 3.08285714285714,
    79: 3.09428571428571,
    190: 3.09428571428571,
    275: 3.09428571428571,
    28: 3.09714285714286,
    36: 3.09714285714286,
    96: 3.09714285714286,
    70: 3.10285714285714,
    288: 3.10285714285714,
    308: 3.12857142857143,
    157: 3.13142857142857,
    103: 3.13714285714286,
    239: 3.14571428571429,
    337: 3.14571428571429,
    175: 3.15142857142857,
    304: 3.15142857142857,
    305: 3.15714285714286,
    330: 3.16,
    348: 3.16285714285714,
    284: 3.16571428571429,
    126: 3.18,
    270: 3.18285714285714,
    44: 3.19714285714286,
    56: 3.2,
    325: 3.20285714285714,
    40: 3.20571428571429,
    92: 3.20571428571429,
    336: 3.20571428571429,
    282: 3.22,
    343: 3.22,
    64: 3.22285714285714,
    203: 3.22571428571429,
    265: 3.24,
    37: 3.24285714285714,
    233: 3.24285714285714,
    77: 3.24857142857143,
    53: 3.26,
    73: 3.26571428571429,
    50: 3.27142857142857,
    346: 3.27142857142857,
    318: 3.27714285714286,
    189: 3.29714285714286,
    51: 3.30571428571429,
    68: 3.30571428571429,
    311: 3.30857142857143,
    27: 3.31428571428571,
    6: 3.32,
    118: 3.32,
    145: 3.33142857142857,
    82: 3.33428571428571,
    229: 3.33714285714286,
    350: 3.34,
    74: 3.34857142857143,
    114: 3.35714285714286,
    250: 3.35714285714286,
    208: 3.36,
    269: 3.36,
    108: 3.36285714285714,
    262: 3.36857142857143,
    244: 3.37142857142857,
    252: 3.37142857142857,
    351: 3.37428571428571,
    59: 3.37714285714286,
    98: 3.38,
    123: 3.38,
    87: 3.38285714285714,
    10: 3.38571428571429,
    29: 3.38857142857143,
    26: 3.39428571428571,
    112: 3.39714285714286,
    281: 3.40571428571429,
    16: 3.41142857142857,
    255: 3.41142857142857,
    88: 3.41714285714286,
    151: 3.42285714285714,
    7: 3.42571428571429,
    260: 3.42857142857143,
    257: 3.43714285714286,
    307: 3.43714285714286,
    80: 3.44,
    204: 3.44,
    137: 3.44285714285714,
    241: 3.44571428571429,
    225: 3.44857142857143,
    117: 3.45142857142857,
    206: 3.45428571428571,
    65: 3.45714285714286,
    178: 3.45714285714286,
    243: 3.45714285714286,
    32: 3.46,
    63: 3.47428571428571,
    253: 3.47428571428571,
    5: 3.48,
    158: 3.48857142857143,
    224: 3.48857142857143,
    297: 3.50285714285714,
    132: 3.50571428571429,
    39: 3.50857142857143,
    329: 3.50857142857143,
    62: 3.51142857142857,
    99: 3.51428571428571,
    143: 3.51714285714286,
    23: 3.52,
    131: 3.52285714285714,
    9: 3.54571428571429,
    133: 3.54857142857143,
    256: 3.54857142857143,
    48: 3.56285714285714,
    292: 3.56571428571429,
    148: 3.56857142857143,
    191: 3.56857142857143,
    279: 3.57142857142857,
    302: 3.57142857142857,
    186: 3.58285714285714,
    259: 3.58571428571429,
    119: 3.6,
    86: 3.61142857142857,
    179: 3.61714285714286,
    236: 3.61714285714286,
    232: 3.63428571428571,
    116: 3.64,
    147: 3.66,
    192: 3.66285714285714,
    141: 3.66571428571429,
    248: 3.67714285714286,
    309: 3.70285714285714,
    333: 3.70571428571429,
    152: 3.71714285714286,
    43: 3.72571428571429,
    164: 3.73428571428571,
    197: 3.73714285714286,
    30: 3.76285714285714,
    316: 3.77428571428571,
    319: 3.77428571428571,
    201: 3.79142857142857,
    120: 3.80571428571429,
    300: 3.80857142857143,
    142: 3.82857142857143,
    144: 3.84285714285714,
    155: 3.84285714285714,
    230: 3.84285714285714,
    8: 3.86857142857143,
    181: 3.86857142857143,
    31: 3.87714285714286,
    321: 3.88,
    202: 3.9,
    14: 3.90857142857143,
    156: 3.91428571428571,
    183: 3.92,
    273: 3.92,
    24: 3.92285714285714,
    91: 3.98,
    109: 3.98571428571429,
    200: 3.98571428571429,
    274: 3.98571428571429,
    163: 3.99428571428571,
    242: 4.0,
    58: 4.03714285714286,
    69: 4.06285714285714,
    247: 4.06285714285714,
    46: 4.06857142857143,
    231: 4.10285714285714,
    271: 4.14,
    1: 4.14857142857143,
    106: 4.17428571428571,
    210: 4.18,
    41: 4.18857142857143,
    258: 4.33714285714286,
    182: 4.34571428571429,
    42: 4.48,
    213: 4.73714285714286,
}

df, s_mean = calibrate_by_season_notebook(
    teams_path="Teams2016.txt",
    games_glob="Games2016.txt",
    avg_spl=avg_spl,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
    plot_dir="plots"
)

display(df)
print("Weighted mean s =", s_mean)


[2015] best s=6.250, logloss=0.501619, predict=75.38%

Weighted mean s = 6.25


,year,best_s,logloss,n_games,predictability,weighted
0,2015,6.25,0.501619,5369,75.377165,33556.25


Weighted mean s = 6.25


In [18]:
# ------------------------------------------------------------
# calibrate_in_notebook.py  (with Colley rankings print + s check)
# ------------------------------------------------------------
# - EXACT Colley (matches your original weighting & SPL lookups)
# - Prints top-N Colley rankings per season
# - Verifies best s by recomputing log-loss & accuracy
# ------------------------------------------------------------

import glob
import json
import math
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_N = 25          # how many teams to show per season
save_rankings_csv = False  # set True to save full rankings as CSV per season

# ---------- Utility functions ----------
def load_teams(teams_path: str) -> pd.DataFrame:
    """Parse teams file like '   1, Abilene_Chr' -> DataFrame (0-based ids)."""
    with open(teams_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.read().splitlines()

    rows = []
    for ln in lines:
        m = re.match(r"^\s*(\d+)\s*,\s*(.+?)\s*$", ln.strip())
        if m:
            one_based = int(m.group(1))
            rows.append((one_based - 1, m.group(2)))

    if not rows:
        raise ValueError("Could not parse teams file — expected '1, TeamName' style lines")

    df = pd.DataFrame(rows, columns=["team_id", "team_name"])
    return df.sort_values("team_id", ignore_index=True)


def read_games(path: str) -> pd.DataFrame:
    """Games file: 8 columns, no header."""
    df = pd.read_csv(path, header=None)
    if df.shape[1] < 8:
        raise ValueError(f"{path}: expected 8 columns, got {df.shape[1]}")
    return df


def year_from_games(df: pd.DataFrame) -> int:
    """Infer season year from column 1 (YYYYMMDD)."""
    return int(df.iloc[0, 1]) // 10000


def logistic(x: np.ndarray) -> np.ndarray:
    """Logistic function with clipping for numerical safety."""
    x = np.clip(x, -60, 60)
    return 1.0 / (1.0 + np.exp(-x))


# ---------- Colley method (EXACT version) ----------
def compute_colley_exact(
    games: pd.DataFrame,
    num_teams: int,
    avg_spl: dict,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
) -> tuple[np.ndarray, float]:
    """
    EXACT port of your original Colley:
      - Uses ceil(...) - 1 segment index (no clipping)
      - Sets gameWeight only when WINNER's OPPONENT is in avg_spl
      - Looks up avg_spl by ZERO-BASED team id (matches your script)
      - Same matrix & RHS updates
    """
    colleyMatrix = 2 * np.diag(np.ones(num_teams))
    b = np.ones(num_teams)

    def weight_avg_spl(x: int) -> float:
        return 3 * math.log10(1 / (avg_spl[x] - 2)) + 1.75

    numGames = len(games)
    dayBeforeSeason = games.iloc[0, 0] - 1
    lastDayOfSeason = games.iloc[numGames - 1, 0]

    for i in range(numGames):
        team1ID = int(games.iloc[i, 2]) - 1
        team1Score = float(games.iloc[i, 4])
        team2ID = int(games.iloc[i, 5]) - 1
        team2Score = float(games.iloc[i, 7])
        currentDay = float(games.iloc[i, 0])

        # time weighting
        if useWeighting:
            numberSegments = len(segmentWeighting)
            weightIndex = math.ceil(
                numberSegments * ((currentDay - dayBeforeSeason) / (lastDayOfSeason - dayBeforeSeason))
            ) - 1
            timeWeight = float(segmentWeighting[weightIndex])
        else:
            timeWeight = 1.0

        # EXACT winner/opponent rule + ZERO-BASED avg_spl lookup
        used = False
        if team1Score > team2Score and (team2ID+1 in avg_spl):
            gameWeight = weight_avg_spl(team2ID+1) * timeWeight
            used = True
        elif team2Score > team1Score and (team1ID+1 in avg_spl):
            gameWeight = weight_avg_spl(team1ID+1) * timeWeight
            used = True

        if not used:
            continue

        # Colley matrix updates
        colleyMatrix[team1ID, team2ID] -= gameWeight
        colleyMatrix[team2ID, team1ID] -= gameWeight
        colleyMatrix[team1ID, team1ID] += gameWeight
        colleyMatrix[team2ID, team2ID] += gameWeight

        # RHS
        if team1Score > team2Score:
            b[team1ID] += 0.5 * gameWeight
            b[team2ID] -= 0.5 * gameWeight
        elif team2Score > team1Score:
            b[team1ID] -= 0.5 * gameWeight
            b[team2ID] += 0.5 * gameWeight
        # (ties add 0, same as your code)

    r = np.linalg.solve(colleyMatrix, b)

    # Predictability (EXACT)
    correct = 0
    for i in range(numGames):
        t1 = int(games.iloc[i, 2]) - 1
        s1 = float(games.iloc[i, 4])
        t2 = int(games.iloc[i, 5]) - 1
        s2 = float(games.iloc[i, 7])
        if (s1 > s2 and r[t1] > r[t2]) or (s2 > s1 and r[t2] > r[t1]) or (s1 == s2 and r[t1] == r[t2]):
            correct += 1
    predictability = 100.0 * correct / numGames if numGames else 0.0
    return r, predictability


def build_colley_rankings(r: np.ndarray, teams_df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame of rankings sorted by rating desc."""
    iSort = np.argsort(-r)
    return pd.DataFrame({
        "rank": np.arange(1, len(r) + 1, dtype=int),
        "rating": r[iSort],
        "team_name": teams_df.loc[iSort, "team_name"].values,
        "team_id": iSort,
    })


# ---------- Fit logistic scale s ----------
def fit_scale_for_games(
    r: np.ndarray,
    games: pd.DataFrame,
    s_min=1.0,
    s_max=16.0,
    steps=301,
):
    """Find best logistic scale s minimizing log-loss."""
    t1 = games.iloc[:, 2].to_numpy(dtype=int) - 1
    s1 = games.iloc[:, 4].to_numpy(dtype=float)
    t2 = games.iloc[:, 5].to_numpy(dtype=int) - 1
    s2 = games.iloc[:, 7].to_numpy(dtype=float)
    y = (s1 > s2).astype(float) + 0.5 * (s1 == s2)

    dR = r[t1] - r[t2]
    s_values = np.linspace(s_min, s_max, steps)
    losses = []

    for s in s_values:
        p = logistic(dR * s)
        p = np.clip(p, 1e-9, 1 - 1e-9)
        loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        losses.append(loss)

    best_idx = int(np.argmin(losses))
    return float(s_values[best_idx]), float(losses[best_idx]), s_values, np.array(losses)


def recompute_logloss(r, games_df, s):
    """Recheck log-loss & accuracy for a given s (sanity check)."""
    t1 = games_df.iloc[:, 2].to_numpy(int) - 1
    s1 = games_df.iloc[:, 4].to_numpy(float)
    t2 = games_df.iloc[:, 5].to_numpy(int) - 1
    s2 = games_df.iloc[:, 7].to_numpy(float)
    y = (s1 > s2).astype(float) + 0.5 * (s1 == s2)

    dR = r[t1] - r[t2]
    p = logistic(dR * s)
    p = np.clip(p, 1e-9, 1 - 1e-9)

    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    acc  = np.mean(((p >= 0.5) & (y == 1.0)) | ((p < 0.5) & (y == 0.0)) | (y == 0.5))
    return loss, acc


# ---------- Main calibration workflow ----------
def calibrate_by_season_notebook(
    teams_path: str,
    games_glob: str,
    avg_spl: dict | str,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
    s_min=1.0,
    s_max=16.0,
    steps=301,
    plot_dir=None,
):
    """
    Loop seasons matched by games_glob:
      - EXACT Colley ratings
      - print top-N rankings
      - fit best s
      - verify s via recompute_logloss
    Returns summary DataFrame + weighted mean s.
    """
    teams_df = load_teams(teams_path)
    num_teams = len(teams_df)

    # avg_spl: accept dict OR JSON path
    if isinstance(avg_spl, str):
        with open(avg_spl, "r") as f:
            raw = json.load(f)
        avg_spl = {int(k): float(v) for k, v in raw.items()}
    elif isinstance(avg_spl, dict):
        avg_spl = {int(k): float(v) for k, v in avg_spl.items()}
    else:
        raise ValueError("avg_spl must be dict or path to JSON file")

    files = sorted(glob.glob(games_glob))
    if not files:
        raise FileNotFoundError(f"No files matched {games_glob}")

    results = []
    if plot_dir:
        os.makedirs(plot_dir, exist_ok=True)

    for gpath in files:
        games = read_games(gpath)
        yr = year_from_games(games)+1

        # ---- EXACT Colley ----
        r, predict = compute_colley_exact(
            games,
            num_teams,
            avg_spl,
            segmentWeighting=segmentWeighting,
            useWeighting=useWeighting,
        )

        # Build & optionally save rankings
        rankings_df = build_colley_rankings(r, teams_df)

        print(f"\n===== {yr} — Colley Top {TOP_N} =====")
        display(rankings_df.head(TOP_N))

        if save_rankings_csv:
            rankings_df.to_csv(f"colley_rankings_{yr}.csv", index=False)

        # ---- Fit s ----
        best_s, best_loss, s_vals, losses = fit_scale_for_games(
            r, games, s_min=s_min, s_max=s_max, steps=steps
        )

        # Verify s
        loss_chk, acc_chk = recompute_logloss(r, games, best_s)

        print(f"[{yr}] best s={best_s:.3f}, logloss={best_loss:.6f}, predict={predict:.2f}%")
        print(f"   sanity-check at s={best_s:.3f}: logloss={loss_chk:.6f}, accuracy={acc_chk*100:.2f}%")

        # Optional plot
        if plot_dir:
            plt.figure()
            plt.plot(s_vals, losses, "-")
            plt.axvline(best_s, color="r", linestyle="--", label=f"best s={best_s:.2f}")
            plt.xlabel("scale (s)")
            plt.ylabel("log-loss")
            plt.title(f"{yr} calibration curve")
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f"scale_{yr}.png"), dpi=150)
            plt.close()

        results.append((yr, best_s, best_loss, len(games), predict))

    df = pd.DataFrame(results, columns=["year", "best_s", "logloss", "n_games", "predictability"])
    df["weighted"] = df["best_s"] * df["n_games"]
    weighted_mean = df["weighted"].sum() / df["n_games"].sum()
    print("\nWeighted mean s =", round(weighted_mean, 3))

    return df.sort_values("year"), weighted_mean

 
avg_spl = {
  268: 3.487603305785124,
  30: 3.209366391184573,
  45: 3.068870523415978,
  238: 3.2920110192837466,
  16: 3.462809917355372,
  291: 3.278236914600551,
  322: 3.1983471074380163,
  89: 3.1019283746556474,
  137: 2.909090909090909,
  150: 3.484848484848485,
  223: 2.5234159779614327,
  145: 3.5674931129476586,
  200: 2.763085399449036,
  255: 3.1818181818181817,
  36: 3.2176308539944904,
  271: 3.1101928374655645,
  325: 2.647382920110193,
  46: 3.25068870523416,
  337: 3.206611570247934,
  311: 3.0,
  343: 3.393939393939394,
  260: 3.071625344352617,
  202: 2.6584022038567494,
  206: 3.37465564738292,
  56: 2.757575757575758,
  211: 3.303030303030303,
  32: 3.465564738292011,
  69: 3.256198347107438,
  227: 2.440771349862259,
  314: 3.077134986225895,
  99: 3.325068870523416,
  192: 3.74931129476584,
  276: 3.371900826446281,
  28: 3.347107438016529,
  15: 2.7327823691460056,
  3: 3.1349862258953167,
  106: 2.564738292011019,
  20: 2.581267217630854,
  10: 2.556473829201102,
  41: 4.25068870523416,
  26: 2.8484848484848486,
  49: 5.09366391184573,
  34: 2.68870523415978,
  180: 3.212121212121212,
  55: 2.7355371900826446,
  81: 3.716253443526171,
  76: 2.363636363636364,
  158: 3.2947658402203857,
  146: 3.0798898071625342,
  77: 2.8650137741046837,
  115: 2.46831955922865,
  130: 4.1652892561983474,
  121: 2.515151515151515,
  78: 3.765840220385675,
  128: 2.5068870523415976,
  186: 5.622589531680441,
  134: 2.479338842975207,
  117: 3.5895316804407718,
  139: 2.9173553719008263,
  8: 3.28099173553719,
  152: 2.4986225895316805,
  184: 3.465564738292011,
  163: 2.5289256198347108,
  159: 3.446280991735537,
  173: 2.435261707988981,
  51: 3.330578512396694,
  199: 2.5482093663911844,
  333: 3.553719008264463,
  210: 2.556473829201102,
  84: 2.9834710743801653,
  241: 2.4242424242424243,
  293: 3.5702479338842976,
  282: 2.669421487603306,
  194: 3.765840220385675,
  270: 2.6143250688705235,
  294: 3.3085399449035813,
  212: 3.15702479338843,
  272: 2.7520661157024797,
  292: 2.8677685950413223,
  142: 3.5950413223140494,
  295: 2.6391184573002757,
  92: 4.12396694214876,
  318: 2.5289256198347108,
  248: 3.6170798898071626,
  336: 2.6143250688705235,
  21: 3.696969696969697,
  340: 3.0192837465564737,
  70: 3.8292011019283754,
  346: 2.5950413223140494,
  59: 4.115702479338843,
  351: 2.5509641873278235,
  249: 3.184573002754821,
  50: 2.4545454545454546,
  44: 3.5041322314049586,
  239: 2.81267217630854,
  126: 3.4765840220385678,
  209: 3.137741046831956,
  2: 3.5867768595041323,
  37: 3.115702479338843,
  123: 3.74931129476584,
  108: 2.837465564738292,
  63: 3.9807162534435263,
  48: 2.5454545454545454,
  13: 4.62534435261708,
  328: 2.727272727272727,
  7: 4.223140495867769,
  52: 3.68870523415978,
  344: 3.43801652892562,
  53: 2.798898071625344,
  253: 3.1487603305785123,
  68: 2.5482093663911844,
  280: 3.7327823691460056,
  287: 2.652892561983471,
  71: 3.6170798898071626,
  228: 2.8980716253443526,
  330: 3.68870523415978,
  302: 3.074380165289256,
  80: 3.2975206611570247,
  245: 2.7796143250688705,
  88: 3.7024793388429753,
  171: 3.319559228650138,
  87: 4.008264462809917,
  91: 2.399449035812672,
  274: 2.975206611570248,
  281: 2.5730027548209367,
  95: 3.242424242424242,
  172: 3.1542699724517904,
  9: 3.1542699724517904,
  101: 2.6914600550964187,
  196: 3.5785123966942147,
  110: 3.4545454545454546,
  160: 3.43801652892562,
  357: 2.4242424242424243,
  114: 3.81267217630854,
  304: 3.556473829201102,
  122: 2.8264462809917354,
  90: 2.834710743801653,
  125: 3.165289256198347,
  133: 2.9807162534435263,
  221: 3.041322314049587,
  57: 3.234159779614325,
  154: 3.765840220385675,
  155: 2.716253443526171,
  47: 4.15426997245179,
  335: 2.5482093663911844,
  166: 4.509641873278237,
  98: 2.768595041322314,
  169: 3.8292011019283754,
  167: 2.382920110192837,
  178: 2.490358126721763,
  104: 3.440771349862259,
  19: 3.3057851239669422,
  174: 2.391184573002755,
  181: 3.184573002754821,
  324: 2.997245179063361,
  6: 3.658402203856749,
  94: 2.68870523415978,
  193: 3.471074380165289,
  230: 3.608815426997245,
  207: 3.900826446280992,
  234: 2.575757575757576,
  244: 3.269972451790633,
  348: 2.8980716253443526,
  236: 3.418732782369146,
  240: 2.831955922865014,
  43: 3.2920110192837466,
  363: 3.1101928374655645,
  243: 3.4986225895316805,
  296: 2.721763085399449,
  254: 3.669421487603306,
  261: 2.6556473829201104,
  283: 2.8539944903581267,
  284: 2.5041322314049586,
  306: 3.184573002754821,
  231: 2.669421487603306,
  24: 3.564738292011019,
  197: 2.8677685950413223,
  262: 4.121212121212121,
  273: 3.176308539944904,
  277: 3.5289256198347108,
  297: 2.37741046831956,
  100: 3.201101928374656,
  103: 2.5482093663911844,
  299: 3.68870523415978,
  222: 2.559228650137741,
  300: 2.5867768595041323,
  214: 3.1046831955922864,
  27: 3.5041322314049586,
  307: 3.060606060606061,
  305: 3.2479338842975207,
  29: 2.7520661157024797,
  264: 3.4765840220385678,
  144: 2.7851239669421486,
  334: 3.236914600550964,
  338: 2.7024793388429758,
  140: 3.757575757575758,
  354: 2.553719008264463,
  358: 2.90633608815427,
  224: 2.6997245179063363,
  352: 3.787878787878788,
  33: 3.721763085399449,
  225: 3.482093663911846,
  39: 2.8567493112947657,
  62: 3.2451790633608817,
  4: 2.363636363636364,
  321: 3.325068870523416,
  164: 3.071625344352617,
  201: 3.9173553719008254,
  362: 2.6942148760330578,
  310: 4.449035812672176,
  266: 2.975206611570248,
  285: 3.4931129476584024,
  161: 2.564738292011019,
  290: 3.426997245179064,
  72: 2.8650137741046837,
  275: 3.5371900826446283,
  217: 2.534435261707989,
  143: 3.652892561983471,
  127: 2.597796143250689,
  83: 3.774104683195592,
  317: 2.6115702479338845,
  301: 2.479338842975207,
  176: 2.4738292011019283,
  147: 3.490358126721763,
  177: 2.5234159779614327,
  350: 3.765840220385675,
  138: 2.391184573002755,
  360: 3.1790633608815426,
  252: 3.0275482093663912,
  165: 3.077134986225895,
  279: 2.68870523415978,
  64: 3.303030303030303,
  259: 2.746556473829201,
  38: 3.1955922865013773,
  246: 3.0991735537190084,
  93: 3.330578512396694,
  35: 2.559228650137741,
  42: 3.8457300275482096,
  135: 2.696969696969697,
  204: 3.8650137741046833,
  213: 2.7355371900826446,
  86: 3.253443526170799,
  11: 2.7300275482093666,
  119: 3.823691460055096,
  303: 2.5482093663911844,
  23: 3.4765840220385678,
  347: 2.743801652892562,
  312: 3.426997245179064,
  232: 3.1818181818181817,
  258: 2.71900826446281,
  315: 2.8484848484848486,
  14: 2.4931129476584024,
  124: 2.5371900826446283,
  269: 3.487603305785124,
  257: 3.716253443526171,
  339: 2.746556473829201,
  40: 3.393939393939394,
  329: 2.6170798898071626,
  112: 2.9366391184573004,
  61: 2.5922865013774103,
  265: 3.1460055096418733,
  102: 2.900826446280992,
  185: 3.765840220385675,
  25: 2.6143250688705235,
  220: 3.25068870523416,
  250: 2.650137741046832,
  345: 3.793388429752066,
  58: 2.6115702479338845,
  219: 2.74931129476584,
  289: 3.661157024793389,
  105: 2.7410468319559227,
  175: 2.606060606060606,
  226: 3.738292011019284,
  156: 2.771349862258953,
  319: 3.8732782369146,
  17: 2.303030303030303,
  132: 3.013774104683196,
  205: 3.8292011019283754,
  195: 3.716253443526171,
  198: 3.256198347107438,
  288: 4.099173553719008,
  73: 3.62534435261708,
  131: 3.256198347107438,
  118: 3.512396694214876,
  191: 2.878787878787879,
  183: 3.4545454545454546,
  216: 3.038567493112948,
  353: 3.1294765840220387,
  237: 5.005509641873278,
  349: 3.633608815426997,
  31: 3.4600550964187327,
  60: 3.049586776859504,
  256: 3.12396694214876,
  157: 3.382920110192837,
  298: 3.721763085399449,
  67: 2.90633608815427,
  313: 2.834710743801653,
  153: 2.9889807162534434,
  182: 3.5867768595041323,
  18: 3.1432506887052343,
  190: 3.512396694214876,
  79: 3.2589531680440773,
  85: 3.203856749311295,
  113: 2.950413223140496,
  187: 3.4049586776859506,
  148: 3.716253443526171,
  242: 3.4159779614325068,
  342: 3.9889807162534434,
  96: 3.7548209366391183,
  65: 4.027548209366391,
  109: 3.2644628099173554,
  129: 3.8953168044077136,
  97: 2.87603305785124,
  22: 2.950413223140496,
  229: 3.4765840220385678,
  189: 2.9366391184573004,
  355: 3.159779614325069,
  116: 3.884297520661157,
  308: 3.303030303030303,
  323: 2.997245179063361,
  286: 3.1487603305785123,
  120: 2.881542699724518,
  233: 3.2589531680440773,
  320: 3.5509641873278235,
  251: 3.743801652892562,
  107: 4.15426997245179,
  218: 3.5867768595041323,
  149: 3.1267217630854,
  208: 3.1515151515151514,
  316: 3.3112947658402203,
  235: 3.352617079889807,
  278: 3.5068870523415976,
  331: 3.071625344352617,
  332: 2.958677685950413,
  170: 3.3856749311294765,
  364: 3.347107438016529,
  309: 3.1790633608815426,
  151: 2.928374655647383,
  326: 3.2148760330578514,
  136: 2.8292011019283745,
  359: 3.115702479338843,
  82: 2.887052341597796,
  75: 3.173553719008265,
  54: 3.581267217630854,
  188: 2.928374655647383,
  1: 3.162534435261708,
  203: 2.9889807162534434,
  356: 3.303030303030303,
  12: 3.413223140495868,
  247: 3.256198347107438,
  66: 3.1955922865013773,
  162: 3.1432506887052343,
  341: 3.517906336088154,
  361: 3.1790633608815426,
  74: 2.6391184573002757,
  267: 3.355371900826446,
  111: 3.0964187327823693,
  263: 3.495867768595041,
  179: 3.088154269972452,
  141: 3.721763085399449,
  168: 3.5426997245179064,
  215: 3.6556473829201104,
  327: 3.0275482093663912,
  5: 4.4986225895316805,
}

df, s_mean = calibrate_by_season_notebook(
    teams_path="NCAA_2025_Teams.txt",
    games_glob="NCAA_2025_Games.txt",
    avg_spl=avg_spl,
    segmentWeighting=(0.5, 1, 2),
    useWeighting=True,
    plot_dir="plots"
)

display(df)
print("Weighted mean s =", s_mean)
# ---------- Example usage in your notebook ----------
# NOTE: you can pass avg_spl as a dict (like you pasted) or as a path to a JSON file.

# avg_spl = {...}  # (your dict here)

# df, s_mean = calibrate_by_season_notebook(
#     teams_path="2015Teams.txt",
#     games_glob="2015Games.txt",     # or "NCAA_*_Games.txt" for many seasons
#     avg_spl=avg_spl,                # dict or "avg_spl_XXXX.json"
#     segmentWeighting=(0.5, 1, 2),
#     useWeighting=True,
#     plot_dir="plots"
# )
# display(df)
# print("Weighted mean s =", s_mean)



===== 2025 — Colley Top 25 =====


,rank,rating,team_name,team_id
0,1,1.175410,Florida,90
1,2,1.135374,Houston,114
2,3,1.127194,Auburn,16
3,4,1.098758,Duke,75
4,5,1.083094,Tennessee,296
5,6,1.047614,Alabama,3
6,7,1.032352,St_John's,280
7,8,1.029399,Michigan_St,173
8,9,0.976366,Michigan,172
9,10,0.968877,Clemson,49


[2025] best s=6.050, logloss=0.508642, predict=74.53%
   sanity-check at s=6.050: logloss=0.508642, accuracy=74.53%

Weighted mean s = 6.05


,year,best_s,logloss,n_games,predictability,weighted
0,2025,6.05,0.508642,5641,74.525793,34128.05


Weighted mean s = 6.050000000000001
